# 39 · seed 21 · PC-CNN (HPCCNNTopDownRaoPCN)

- Same as 39_resizemix50_seed42.ipynb; only SEED = 21 and SAVE_DIR changed.

In [ ]:
import random
from pathlib import Path
import torch

# torch.set_num_threads(8)
# print("PyTorch CPU threads:", torch.get_num_threads())

from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode

from src.dataset.balanced_class_subset import BalancedClassSubset
from src.dataset.tiny_imagenet_dataset import TinyImageNetLoader, get_selected_classes, get_selected_class_names
from src.utils.utils import means, stds, convert_to_rgb, get_device, print_device_info, set_seed, make_generator, count_parameters
from src.utils.plot_graph import show_image, show_mix_samples, plot_training_curves
from src.dataset.resizemix import ResizeMixDataset
from src.model.pc_cnn_top_down_rao_pcn import HPCCNNTopDownRaoPCN
from src.utils.metrics_csv import print_metrics
from src.utils.train_model import TrainModel
from src.utils.checkpoint import CheckpointManager

RESUME = False
SEED = 21
NUM_CLASSES = 200
TRAIN_PER_CLASS = 500
VAL_PER_CLASS = 50
BATCH_SIZE = 64
NUM_WORKERS = 4
EPOCHS = 100
INITIAL_LR = 0.05
MIN_LR = 1e-05
WEIGHT_DECAY = 0.0001
LABEL_SMOOTHING = 0.1
BLOCK3_DROPOUT = 0.15  # Dropout2d after r6, as in the Standard CNN (same mask in every cycle)
BLOCK4_DROPOUT = 0.25  # Dropout2d after r8, as in the Standard CNN (same mask in every cycle)
CLASSIFIER_DROPOUT = 0.35  # dense layer
CLEAN_EVAL_INTERVAL = 5

RESIZEMIX_SCALE = (0.1, 0.8)
RESIZEMIX_PROB = 0.50

# Predictive coding: Rao & Ballard (1999) + PCN (Wen et al., 2018)
INFERENCE_STEPS = 7  # cycles in training and at test (proposal slide 11; Wen et al.: the same number in both)
BOTTOM_UP_RATE = 1.0  # initial a: error correction, in units of the stable step 1 / lambda (Wen et al.: 1.0)
TOP_DOWN_RATE = 0.5  # initial b: move toward the top-down prediction (Wen et al.: 0.5)

TINY_ROOT = Path('../../data/tiny_imagenet/tiny-imagenet-200')
SAVE_DIR = Path('../../checkpoints/pc_cnn_top_down_rao_pcn/39_resizemix50_seed21')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = get_device()

set_seed(SEED)

print('Device:', DEVICE)
print('Dataset root:', TINY_ROOT)
print('DataLoader workers:', NUM_WORKERS)
print('Epochs:', EPOCHS)
print(f'ResizeMix on {RESIZEMIX_PROB:.0%} of images | scale={RESIZEMIX_SCALE} | RandAugment after mixing')
print(f'Predictive coding (Rao & Ballard + PCN): {INFERENCE_STEPS} cycles in training and at test | '
      f'initial rates: bottom-up a {BOTTOM_UP_RATE}, top-down b {TOP_DOWN_RATE} (learned per channel)')
print(f'Dropout: hidden layers {BLOCK3_DROPOUT}, {BLOCK4_DROPOUT} | dense {CLASSIFIER_DROPOUT} | hidden dropout in every cycle (same mask)')

In [ ]:
print_device_info(DEVICE)

## 1. Dataset and transforms

In [ ]:
before_mix_tf = transforms.Compose(
    [transforms.Lambda(convert_to_rgb), transforms.RandomCrop(64, padding=4, padding_mode='reflect'),
     transforms.RandomHorizontalFlip(p=0.5)])
after_mix_tf = transforms.Compose(
    [transforms.RandAugment(num_ops=2, magnitude=5, interpolation=InterpolationMode.BILINEAR), transforms.ToTensor(),
     transforms.Normalize(mean=means, std=stds)])
test_tf = transforms.Compose(
    [transforms.Lambda(convert_to_rgb), transforms.ToTensor(), transforms.Normalize(mean=means, std=stds)])

train_dataset, val_dataset, _, _ = TinyImageNetLoader(seed=SEED, root=TINY_ROOT, train_transform=before_mix_tf,
                                                      test_transform=test_tf).load()
clean_train_dataset, _, _, _ = TinyImageNetLoader(seed=SEED, root=TINY_ROOT, train_transform=test_tf,
                                                  test_transform=test_tf).load()

if not hasattr(clean_train_dataset, 'transform'):
    raise AttributeError('Expected the dataset to expose a transform attribute.')

train_subset = BalancedClassSubset(train_dataset, get_selected_classes(NUM_CLASSES), TRAIN_PER_CLASS, seed=SEED)
clean_train_subset = BalancedClassSubset(clean_train_dataset, get_selected_classes(NUM_CLASSES), TRAIN_PER_CLASS, seed=SEED)
val_subset = BalancedClassSubset(val_dataset, get_selected_classes(NUM_CLASSES), VAL_PER_CLASS, seed=SEED + 1)

assert train_subset.samples == clean_train_subset.samples, 'Clean and augmented training subsets differ.'
assert len(train_subset) == NUM_CLASSES * TRAIN_PER_CLASS
assert len(val_subset) == NUM_CLASSES * VAL_PER_CLASS

CLASS_NAMES = get_selected_class_names(train_dataset, TINY_ROOT, NUM_CLASSES)

print('Train:', len(train_subset), 'Validation:', len(val_subset))

## 2. DataLoaders (macOS / Jupyter safe)

In [ ]:
loader_generator = make_generator(SEED)

loader_options = {'batch_size': BATCH_SIZE, 'num_workers': NUM_WORKERS, 'pin_memory': DEVICE.type == 'cuda',
                  'persistent_workers': NUM_WORKERS > 0, **({'prefetch_factor': 2} if NUM_WORKERS > 0 else {})}

train_mix_dataset = ResizeMixDataset(train_subset, after_mix_tf, scale=RESIZEMIX_SCALE, mix_prob=RESIZEMIX_PROB)
train_loader = DataLoader(train_mix_dataset, shuffle=True, generator=loader_generator, **loader_options)
val_loader = DataLoader(val_subset, shuffle=False, **loader_options)
clean_train_loader = DataLoader(clean_train_subset, shuffle=False, **loader_options)

print('Training batches:', len(train_loader))
print('Validation batches:', len(val_loader))

In [ ]:
# Sample check: plain images (top) and training images with ResizeMix
show_mix_samples(train_dataset, train_subset, train_mix_dataset)

## 3. Model, optimizer and cosine scheduler

In [ ]:
# Model: src/model/pc_cnn_top_down_rao_pcn.py (Standard CNN + predictive coding)

# ==========================================
# MODEL INITIALIZATION
# ==========================================

base_model = HPCCNNTopDownRaoPCN(num_classes=NUM_CLASSES, block3_dropout=BLOCK3_DROPOUT,
                                 block4_dropout=BLOCK4_DROPOUT, classifier_dropout=CLASSIFIER_DROPOUT,
                                 level4_dilation=2, inference_steps=INFERENCE_STEPS,
                                 bottom_up_rate=BOTTOM_UP_RATE, top_down_rate=TOP_DOWN_RATE)
# Validation uses the 7 cycles
base_model.top_down_inference = True

total_params = count_parameters(base_model)
rate_params = sum(p.numel() for p in base_model.bottom_up_rates) + sum(p.numel() for p in base_model.top_down_rates)
print(f"Parameters: {total_params:,} ({total_params / 1e6:.3f}M) = network {total_params - rate_params:,} "
      f"+ update rates {rate_params:,} (the top-down predictions use the conv weights, transposed)")
print(f"Architecture: 8 conv (2+2+2+2, conv8 dilation 2) | BatchNorm x9 (8 conv + dense) | LeakyReLU 0.05 | MaxPool x3 | Dropout2d 0.15 (r6), 0.25 (r8) in every cycle | GAP 512 → dense 512 (dropout 0.35) → {NUM_CLASSES}")

model = base_model.to(DEVICE)



# LOSS, OPTIMIZER & SCHEDULER
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.SGD(model.parameters(), lr=INITIAL_LR, momentum=0.9, weight_decay=WEIGHT_DECAY, nesterov=True)

TOTAL_STEPS = EPOCHS * len(train_loader)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TOTAL_STEPS, eta_min=MIN_LR)

# SANITY CHECK
model.eval()
with torch.no_grad():
    dummy = torch.randn(2, 3, 64, 64, device=DEVICE)
    logits, features = model(dummy, return_features=True)

print("Logits shape:", logits.shape)
print("Feature shape:", features.shape)
model.train()

## 4. Checkpoints and resume

In [ ]:
# Checkpoints and resume (src/utils/checkpoint.py); RESUME needs the same config
CONFIG = dict(num_classes=NUM_CLASSES, batch_size=BATCH_SIZE, epochs=EPOCHS, train_per_class=TRAIN_PER_CLASS,
              val_per_class=VAL_PER_CLASS, classifier='8conv_v2_gap_dense_fc', block3_dropout=BLOCK3_DROPOUT,
              block4_dropout=BLOCK4_DROPOUT, classifier_dropout=CLASSIFIER_DROPOUT,
              augmentation='resizemix_half_of_images_randaugment2m5_after', resizemix_scale=RESIZEMIX_SCALE,
              resizemix_prob=RESIZEMIX_PROB, model='HPCCNNTopDownRaoPCN', predictive_coding='pcn_tied_full_cycles_every_conv_layer_dropout_every_cycle_v2', inference_steps=INFERENCE_STEPS,
              bottom_up_rate=BOTTOM_UP_RATE, top_down_rate=TOP_DOWN_RATE)

ckpt = CheckpointManager(SAVE_DIR, model, optimizer, scheduler, CONFIG, DEVICE, generators={'loader': loader_generator})
history, best_acc, best_epoch, start_epoch = ckpt.start(RESUME)
BEST_PATH = ckpt.best_path

## 5. Training and validation

In [ ]:
# Training: 7 cycles, loss after the last one, backprop through all cycles. Writes metrics.csv.
trainer = TrainModel(model, optimizer, scheduler, criterion, DEVICE)
best_acc, best_epoch = trainer.fit(train_loader, val_loader, clean_train_loader, EPOCHS, history, SAVE_DIR,
                                   ckpt.save_latest, start_epoch=start_epoch, best_acc=best_acc, best_epoch=best_epoch,
                                   clean_eval_interval=CLEAN_EVAL_INTERVAL,
                                   best_info={'class_names': CLASS_NAMES,
                                              'experiment': '39 HPCCNNTopDownRaoPCN: tied PCN (Wen et al. 2018) = Rao & Ballard tied predictive coding at every conv layer, 7 full cycles in training and test, all weights learned by backprop through the cycles, hidden dropout 0.15/0.25 with the same mask in every cycle, ResizeMix on half of images, RandAugment after mixing'})

In [ ]:
# All metrics per epoch (from metrics.csv), then best and mean val of the last 5 epochs
print_metrics(SAVE_DIR / 'metrics.csv')

## 6. Training and validation curves (validation after 7 cycles)

In [ ]:
plot_training_curves(history, train_label='Train (HPCCNNTopDownRaoPCN, 7 cycles, 50% ResizeMix)')

best = ckpt.load_best()
model.eval()

clean_train_loss, clean_train_acc = trainer.run_epoch(clean_train_loader)
clean_val_loss, clean_val_acc = trainer.run_epoch(val_loader)

print(
    f'Best checkpoint: clean train {clean_train_acc:.2f}%, clean val {clean_val_acc:.2f}%, gap {clean_train_acc - clean_val_acc:.2f} pp')

## 7. Accuracy and prediction error per cycle

Final model, clean images. Cycle 0 = one pass. The prediction error should go down over the cycles.

In [ ]:
latest = torch.load(SAVE_DIR / 'latest_checkpoint.pth', map_location='cpu', weights_only=False)
model.load_state_dict(latest['model'])
model.eval()
model.top_down_inference = True
print(f"Final model: epoch {latest['epoch']}")


# Accuracy (%) after every cycle 0..steps over a loader
@torch.no_grad()
def accuracy_by_step(loader, steps=INFERENCE_STEPS):
    correct, total = torch.zeros(steps + 1), 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        for step, logits in enumerate(model.iterative_inference(images, steps)):
            correct[step] += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return (correct * 100 / total).tolist()


# Total prediction error over the 8 layers after every cycle (should go down)
@torch.no_grad()
def prediction_error_by_step(loader, steps=INFERENCE_STEPS):
    errors = torch.zeros(steps + 1)
    for images, _ in loader:
        images = images.to(DEVICE)
        for step in range(steps + 1):
            errors[step] += sum(model.prediction_errors(images, step)[0])
    return errors.tolist()


# Relative error per layer after one pass (0 = perfect)
@torch.no_grad()
def relative_prediction_errors(loader):
    error_sum, norm_sum = torch.zeros(len(model.conv_layers)), torch.zeros(len(model.conv_layers))
    for images, _ in loader:
        errors, norms = model.prediction_errors(images.to(DEVICE), 0)
        error_sum += torch.tensor(errors)
        norm_sum += torch.tensor(norms)
    return (error_sum / norm_sum).sqrt().tolist()


for layer, error in enumerate(relative_prediction_errors(val_loader), start=1):
    print(f'layer {layer} (predicts {"the image" if layer == 1 else f"r{layer - 1}"} from r{layer}): relative error {error:.3f}')

In [ ]:
val_by_step = accuracy_by_step(val_loader)
clean_train_by_step = accuracy_by_step(clean_train_loader)
error_by_step = prediction_error_by_step(val_loader)

print('cycle    | clean val | clean train | prediction error (val)')
for step in range(INFERENCE_STEPS + 1):
    label = 'one pass' if step == 0 else f'{step:>8}'
    print(f'{label} | {val_by_step[step]:9.2f} | {clean_train_by_step[step]:11.2f} | {error_by_step[step]:,.0f}')
print(f'\nClean val: one pass {val_by_step[0]:.2f}% → {INFERENCE_STEPS} cycles {val_by_step[-1]:.2f}% '
      f'({val_by_step[-1] - val_by_step[0]:+.2f}) | prediction error {error_by_step[0]:,.0f} → {error_by_step[-1]:,.0f} '
      f'({(error_by_step[-1] / error_by_step[0] - 1) * 100:+.1f}%)')

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for axis, values, title in zip(axes[:2], (val_by_step, clean_train_by_step), ('Clean val', 'Clean train')):
    axis.plot(range(INFERENCE_STEPS + 1), values, marker='o')
    axis.axhline(values[0], color='gray', linestyle='--', label='cycle 0 (one pass of this network)')
    axis.set(xlabel='cycle', ylabel='accuracy (%)', title=f'{title}: accuracy by cycle')
    axis.grid(alpha=0.3)
    axis.legend()
axes[2].plot(range(INFERENCE_STEPS + 1), error_by_step, marker='o', color='tab:red')
axes[2].set(xlabel='cycle', ylabel='total squared prediction error', title='Clean val: prediction error by cycle')
axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Learned rates and more cycles than trained

Rates b (top-down) and a (bottom-up) and λ per layer, then clean val accuracy for 0–15 cycles (trained with 7).

In [ ]:
# Learned rates per layer (kept >= 0 with ReLU, as in the model)
layers = len(model.bottom_up_rates)
print('layer | top-down rate b: mean (min - max) | bottom-up rate a: mean (min - max) | lambda')
for layer in range(1, layers + 1):
    bottom_up = torch.relu(model.bottom_up_rates[layer - 1].detach()).flatten()
    if layer < layers:
        top_down = torch.relu(model.top_down_rates[layer - 1].detach()).flatten()
        top_down_text = f'{top_down.mean():.3f} ({top_down.min():.3f} - {top_down.max():.3f})'
    else:
        top_down_text = 'none (top layer)'
    print(f'r{layer}    | {top_down_text:33s} | {bottom_up.mean():.3f} ({bottom_up.min():.3f} - {bottom_up.max():.3f})'
          f'                | {model.top_down_curvature[layer - 1]:.1f}')

# Accuracy for 0..15 cycles (trained with 7)
MORE_STEPS = 15
val_more = accuracy_by_step(val_loader, steps=MORE_STEPS)
print('\nClean val by cycle 0..15:', ' '.join(f'{value:.1f}' for value in val_more))
print(f'cycle {INFERENCE_STEPS}: {val_more[INFERENCE_STEPS]:.2f}% | cycle {MORE_STEPS}: {val_more[-1]:.2f}%')